# Step 01_02_05 -- Univariate Census Visualizations: aoe2companion

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_02 -- EDA
**Dataset:** aoe2companion
**Question:** What do the univariate distributions from 01_02_04 look like visually?
**Invariants applied:** #6 (reproducibility -- SQL inlined in artifact),
#7 (no magic numbers), #9 (step scope: visualization only)
**Step scope:** visualization -- reads 01_02_04 JSON artifact and DuckDB (read-only)
**Type:** Read-only -- no DuckDB writes, no new tables, no schema changes

## 0. Imports and DB connection

In [1]:
import json
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rts_predict.common.notebook_utils import (
    get_notebook_db,
    get_reports_dir,
    setup_notebook_logging,
)

logger = setup_notebook_logging()
matplotlib.use("Agg")
plt.style.use("seaborn-v0_8-whitegrid")

In [2]:
db = get_notebook_db("aoe2", "aoe2companion")
con = db.con
print("Connected to aoe2companion DuckDB (read-only)")

Connected to aoe2companion DuckDB (read-only)


In [3]:
reports_dir = get_reports_dir("aoe2", "aoe2companion")
artifacts_dir = reports_dir / "artifacts" / "01_exploration" / "02_eda"
artifacts_dir.mkdir(parents=True, exist_ok=True)
plots_dir = artifacts_dir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
print(f"Artifacts dir: {artifacts_dir}")
print(f"Plots dir: {plots_dir}")

Artifacts dir: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/02_eda
Plots dir: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/02_eda/plots


In [4]:
census_json_path = artifacts_dir / "01_02_04_univariate_census.json"
with open(census_json_path) as f:
    census = json.load(f)
print(f"Loaded 01_02_04 artifact: {len(census)} keys")
print(f"Keys: {sorted(census.keys())}")

Loaded 01_02_04 artifact: 45 keys
Keys: ['boolean_census', 'categorical_profiles', 'colorHex_cardinality', 'constant_fields', 'db_memory_footprint_bytes', 'duration_excluded_rows', 'empty_string_investigation', 'exact_won_null_note', 'field_classification', 'leaderboards_raw_boolean', 'leaderboards_raw_categorical', 'leaderboards_raw_null_census', 'leaderboards_raw_numeric_stats', 'leaderboards_raw_skew_kurtosis', 'leaderboards_raw_temporal', 'leaderboards_raw_total_rows', 'leaderboards_raw_zero_counts', 'match_duration_stats', 'match_structure_by_leaderboard', 'matches_raw_duplicate_check', 'matches_raw_null_census', 'matches_raw_null_cooccurrence', 'matches_raw_numeric_stats', 'matches_raw_skew_kurtosis', 'matches_raw_total_rows', 'matches_raw_zero_counts', 'name_cardinality', 'near_constant_low_cardinality', 'near_constant_ratio_flagged', 'post_game_fields', 'profiles_raw_categorical', 'profiles_raw_null_census', 'profiles_raw_total_rows', 'ratings_raw_duplicate_check', 'ratings_raw

## T02 -- Plot 1: won distribution bar chart

In [5]:
won_dist = pd.DataFrame(census["won_distribution"])
print("=== won_dist (feeds bar chart) ===")
print(won_dist.to_string(index=False))

=== won_dist (feeds bar chart) ===
  won       cnt   pct
False 132150323 47.69
 True 131963175 47.62
 None  12985561  4.69


In [6]:
fig, ax = plt.subplots(figsize=(8, 6))
color_map = {True: '#2ecc71', False: '#e74c3c', None: '#95a5a6'}
labels = []
counts = []
colors = []
for row in census["won_distribution"]:
    label = str(row["won"]) if row["won"] is not None else "NULL"
    labels.append(label)
    counts.append(row["cnt"])
    colors.append(color_map[row["won"]])
bars = ax.bar(labels, counts, color=colors, edgecolor='black', linewidth=0.5)
for bar, cnt, row in zip(bars, counts, census['won_distribution']):
    ax.annotate(
        f"{cnt:,}\n({row['pct']:.2f}%)",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=10,
    )
ax.set_xlabel("won value")
ax.set_ylabel("Row count")
ax.set_title("matches_raw: won Value Distribution")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig(plots_dir / "01_02_05_won_distribution.png", dpi=150)
plt.close()
print("Saved: 01_02_05_won_distribution.png")

Saved: 01_02_05_won_distribution.png


## T03 -- Plot 2: Intra-match consistency stacked bar

In [7]:
# CRITICAL: won_consistency_2row is a single-element list containing ONE dict
# with category names as keys. Do NOT use pd.DataFrame and then access ['category'].
raw = census["won_consistency_2row"][0]
total = raw["total_2row_matches"]
categories = [
    "consistent_complement", "both_true", "both_false",
    "both_null", "one_true_one_null", "one_false_one_null",
]
consistency = pd.DataFrame([
    {"category": cat, "count": raw[cat]} for cat in categories
])
consistency['pct'] = 100.0 * consistency['count'] / total
print("=== won_consistency_2row (feeds stacked bar) ===")
print(f"total_2row_matches: {total:,}")
print(consistency.to_string(index=False))

=== won_consistency_2row (feeds stacked bar) ===
total_2row_matches: 40,062,975
             category    count       pct
consistent_complement 35479656 88.559714
            both_true  2499163  6.238086
           both_false  1899564  4.741445
            both_null     1885  0.004705
    one_true_one_null    66642  0.166343
   one_false_one_null   116065  0.289706


In [8]:
# Compute inconsistency annotation dynamically -- do NOT hardcode any percentage
inconsistent_count = raw["both_true"] + raw["both_false"]
inconsistent_pct = 100.0 * inconsistent_count / total
annotation = (
    f"~{inconsistent_pct:.2f}% of 2-player matches have both_true or both_false "
    f"won labels (empirical noise floor on prediction target accuracy).\n"
    f"Note: this counts only both_true+both_false; other inconsistent categories "
    f"(one_true_one_null, etc.) are shown but not counted here."
)
color_map_consistency = {
    "consistent_complement": "#2ecc71",
    "both_true": "#e67e22",
    "both_false": "#e67e22",
    "both_null": "#95a5a6",
    "one_true_one_null": "#f1c40f",
    "one_false_one_null": "#f1c40f",
}
fig, ax = plt.subplots(figsize=(14, 4))
left = 0.0
for _, row in consistency.iterrows():
    cat = row['category']
    pct = row['pct']
    ax.barh(0, pct, left=left, color=color_map_consistency[cat], label=cat,
            edgecolor='white', linewidth=0.5)
    if pct > 1.0:
        ax.text(
            left + pct / 2, 0, f'{pct:.1f}%',
            ha='center', va='center', fontsize=8, fontweight='bold', color='black'
        )
    left += pct
ax.set_xlim(0, 100)
ax.set_yticks([])
ax.set_xlabel("Percentage of 2-player matches")
ax.set_title("matches_raw: Intra-Match won Consistency (2-player matches)")
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=9)
ax.text(
    0.01, -0.35, annotation,
    transform=ax.transAxes, fontsize=8, va='top',
    style='italic', color='#555555'
)
plt.tight_layout()
plt.savefig(plots_dir / "01_02_05_won_consistency.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: 01_02_05_won_consistency.png")

Saved: 01_02_05_won_consistency.png


## T04 -- Plots 3-5: Leaderboard, Civ top-30, Map top-30 horizontal bars

In [9]:
# I7: All 30 entries from the categorical_profiles artifact are plotted
# (no arbitrary cutoff). For civ, rank 30 (slavs) still represents 1.68%
# of rows. For map, rank 30 (rm_baltic) represents 0.42%.
# The artifact's LIMIT 30 in the source SQL is the data boundary here.
leaderboard_data = pd.DataFrame(census["categorical_profiles"]["leaderboard"])
print("=== leaderboard (feeds bar chart) ===")
print(leaderboard_data.to_string(index=False))

=== leaderboard (feeds bar chart) ===
                       value       cnt   pct
                     rm_team 102711164 37.07
                    unranked  78254732 28.24
                      rm_1v1  53694523 19.38
                  qp_rm_team  19707155  7.11
                   qp_rm_1v1   7377276  2.66
                     unknown   3782726  1.37
             rm_team_console   3472383  1.25
                     ew_team   3356610  1.21
                      ew_1v1   1943971  0.70
              rm_1v1_console   1600477  0.58
        ew_1v1_redbullwololo    594890  0.21
                     dm_team    193400  0.07
                   qp_br_ffa    167984  0.06
                      dm_1v1     94508  0.03
                   qp_ew_1v1     49477  0.02
                  qp_ew_team     44716  0.02
                    ror_team     23308  0.01
                     ror_1v1     15775  0.01
              ew_1v1_console      7522  0.00
             ew_team_console      5812  0.00
ew_1v1_redbullwol

In [10]:
leaderboard_data_sorted = leaderboard_data.sort_values('cnt', ascending=True)
fig, ax = plt.subplots(figsize=(10, max(4, len(leaderboard_data_sorted) * 0.4)))
bars = ax.barh(leaderboard_data_sorted['value'], leaderboard_data_sorted['cnt'],
               color='#3498db', edgecolor='black', linewidth=0.3)
for bar, cnt in zip(bars, leaderboard_data_sorted['cnt']):
    ax.text(
        bar.get_width() + bar.get_width() * 0.01, bar.get_y() + bar.get_height() / 2,
        f'{cnt:,}', va='center', ha='left', fontsize=9
    )
ax.set_xlabel("Row count")
ax.set_title("matches_raw: leaderboard Distribution")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig(plots_dir / "01_02_05_leaderboard_distribution.png", dpi=150)
plt.close()
print("Saved: 01_02_05_leaderboard_distribution.png")

Saved: 01_02_05_leaderboard_distribution.png


In [11]:
civ_data = pd.DataFrame(census["categorical_profiles"]["civ"])
print(f"=== civ top-{len(civ_data)} (feeds bar chart) ===")
print(civ_data.to_string(index=False))

=== civ top-30 (feeds bar chart) ===
      value      cnt  pct
     franks 15745563 5.68
    mongols 12320535 4.45
    britons 11814240 4.26
   persians 10762563 3.88
    spanish 10455030 3.77
      khmer 10350344 3.74
       huns  9475018 3.42
    teutons  8727584 3.15
      goths  8453800 3.05
      turks  7535597 2.72
 ethiopians  7308930 2.64
 byzantines  7231564 2.61
    magyars  7207859 2.60
lithuanians  6918321 2.50
 portuguese  6719457 2.42
     mayans  6308353 2.28
   japanese  6291559 2.27
 vietnamese  6127900 2.21
     cumans  6126789 2.21
      celts  6046451 2.18
    chinese  5752126 2.08
   italians  5700089 2.06
   saracens  5603514 2.02
hindustanis  5472128 1.97
    vikings  5356311 1.93
  bohemians  5162562 1.86
burgundians  4930876 1.78
 bulgarians  4768844 1.72
      poles  4752593 1.72
      slavs  4654425 1.68


In [12]:
civ_data_sorted = civ_data.sort_values('cnt', ascending=True)
fig, ax = plt.subplots(figsize=(10, max(6, len(civ_data_sorted) * 0.35)))
bars = ax.barh(civ_data_sorted['value'], civ_data_sorted['cnt'],
               color='#9b59b6', edgecolor='black', linewidth=0.3)
for bar, cnt in zip(bars, civ_data_sorted['cnt']):
    ax.text(
        bar.get_width() + bar.get_width() * 0.01, bar.get_y() + bar.get_height() / 2,
        f'{cnt:,}', va='center', ha='left', fontsize=7
    )
ax.set_xlabel("Row count")
ax.set_title(f"matches_raw: civ Top {len(civ_data_sorted)} Distribution")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig(plots_dir / "01_02_05_civ_top30.png", dpi=150)
plt.close()
print("Saved: 01_02_05_civ_top30.png")

Saved: 01_02_05_civ_top30.png


In [13]:
map_data = pd.DataFrame(census["categorical_profiles"]["map"])
print(f"=== map top-{len(map_data)} (feeds bar chart) ===")
print(map_data.to_string(index=False))

=== map top-30 (feeds bar chart) ===
              value      cnt   pct
          rm_arabia 54118272 19.53
           rm_arena 46540790 16.80
    rm_black-forest 34421288 12.42
           rm_nomad 18932603  6.83
            unknown 14792062  5.34
      rm_megarandom 14292224  5.16
  rm_amazon_tunnels  7273441  2.62
rm_african_clearing  6568294  2.37
      rm_land_nomad  5476005  1.98
           rm_michi  4299160  1.55
           rm_oasis  3571412  1.29
         rm_hideout  3546713  1.28
       rm_lombardia  3216877  1.16
         rm_coastal  2791794  1.01
        rm_fortress  2549602  0.92
        rm_enclosed  2447904  0.88
         rm_yucatan  2194472  0.79
      rm_four-lakes  2192085  0.79
      rm_golden-pit  1874722  0.68
       rm_gold-rush  1761039  0.64
  rm_coastal_forest  1756840  0.63
    rm_team-islands  1632668  0.59
          rm_steppe  1577116  0.57
        rm_highland  1547839  0.56
      rm_runestones  1503197  0.54
         rm_socotra  1254094  0.45
   rm_mediterranea

In [14]:
map_data_sorted = map_data.sort_values('cnt', ascending=True)
fig, ax = plt.subplots(figsize=(10, max(6, len(map_data_sorted) * 0.35)))
bars = ax.barh(map_data_sorted['value'], map_data_sorted['cnt'],
               color='#1abc9c', edgecolor='black', linewidth=0.3)
for bar, cnt in zip(bars, map_data_sorted['cnt']):
    ax.text(
        bar.get_width() + bar.get_width() * 0.01, bar.get_y() + bar.get_height() / 2,
        f'{cnt:,}', va='center', ha='left', fontsize=7
    )
ax.set_xlabel("Row count")
ax.set_title(f"matches_raw: map Top {len(map_data_sorted)} Distribution")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig(plots_dir / "01_02_05_map_top30.png", dpi=150)
plt.close()
print("Saved: 01_02_05_map_top30.png")

Saved: 01_02_05_map_top30.png


## T05 -- Plots 6-7: rating and ratingDiff histograms

In [15]:
# I7: rating range -1 to 5001 (~5002 range); 5002/100 = ~50 bins --
# adequate resolution for skewness=0.57
rating_hist_df = con.execute('''
    SELECT (FLOOR(rating / 100) * 100)::INTEGER AS bin, COUNT(*) AS cnt
    FROM matches_raw WHERE rating IS NOT NULL
    GROUP BY bin ORDER BY bin
''').df()
print("=== rating histogram bins (feeds histogram plot) ===")
print(rating_hist_df.to_string(index=False))

=== rating histogram bins (feeds histogram plot) ===
 bin      cnt
-100      534
   0    46951
 100    78568
 200   171650
 300   396362
 400   980370
 500  2312309
 600  4769442
 700  8949311
 800 15461489
 900 21956782
1000 26136311
1100 20651170
1200 17804317
1300 14988617
1400 10663652
1500  6197143
1600  3184322
1700  1746394
1800  1030710
1900   631429
2000   394760
2100   300708
2200   212480
2300   143368
2400   109482
2500    64764
2600    35788
2700    20880
2800     9226
2900     3982
3000     1560
3100      641
3200      386
3300      305
3400      146
3500       90
3600       97
3700       35
3800       11
3900       20
4000        7
4100        9
4200        5
4300        2
4400        2
4500        2
4800        2
4900        1
5000        1


In [16]:
# Load skewness/kurtosis from census artifact for 'rating'
sk_df = pd.DataFrame(census["matches_raw_skew_kurtosis"])
rating_sk = sk_df[sk_df['column_name'] == 'rating'].iloc[0]
rating_skewness = rating_sk['skewness']
rating_kurtosis = rating_sk['kurtosis']
# Get null stats
null_df = pd.DataFrame(census["matches_raw_null_census"])
rating_null = null_df[null_df['column_name'] == 'rating'].iloc[0]
total_rows = census["matches_raw_total_rows"]
non_null_count = total_rows - int(rating_null['null_count'])
null_pct = rating_null['null_pct']
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(rating_hist_df['bin'], rating_hist_df['cnt'], width=90,
       color='#3498db', edgecolor='black', linewidth=0.2, alpha=0.85)
ax.set_xlabel("rating (bin width=100)")
ax.set_ylabel("Row count")
ax.set_title(
    f'matches_raw.rating '
    f'(N={non_null_count:,} non-NULL of {total_rows:,} total, '
    f'{null_pct:.2f}% NULL; skew={rating_skewness:.4f}, kurt={rating_kurtosis:.4f})'
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig(plots_dir / "01_02_05_rating_histogram.png", dpi=150)
plt.close()
print("Saved: 01_02_05_rating_histogram.png")

Saved: 01_02_05_rating_histogram.png


In [17]:
# I7: ratingDiff range -174 to +319 (~493 range); 493/10 = ~49 bins
ratingdiff_hist_df = con.execute('''
    SELECT (FLOOR("ratingDiff" / 10) * 10)::INTEGER AS bin, COUNT(*) AS cnt
    FROM matches_raw WHERE "ratingDiff" IS NOT NULL
    GROUP BY bin ORDER BY bin
''').df()
print("=== ratingDiff histogram bins (feeds histogram plot) ===")
print(ratingdiff_hist_df.to_string(index=False))

=== ratingDiff histogram bins (feeds histogram plot) ===
 bin      cnt
-180        3
-170       63
-160      122
-150      210
-140      343
-130      488
-120      844
-110     1768
-100     4027
 -90     8785
 -80    21504
 -70    64908
 -60   327927
 -50  1652959
 -40  1686168
 -30  4088668
 -20 62336278
 -10  6676546
   0 10960154
  10 63519644
  20  5400766
  30  1174754
  40   971421
  50   293782
  60    98434
  70    50569
  80    32054
  90    21705
 100    15830
 110    11876
 120     9031
 130     7546
 140     5192
 150     4401
 160     4243
 170     2584
 180        1
 190        3
 200        4
 210        5
 220        1
 260        3
 310        9


In [18]:
ratingdiff_sk = sk_df[sk_df['column_name'] == 'ratingDiff'].iloc[0]
ratingdiff_skewness = ratingdiff_sk['skewness']
ratingdiff_kurtosis = ratingdiff_sk['kurtosis']
ratingdiff_null = null_df[null_df['column_name'] == 'ratingDiff'].iloc[0]
ratingdiff_non_null = total_rows - int(ratingdiff_null['null_count'])
ratingdiff_null_pct = ratingdiff_null['null_pct']
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(ratingdiff_hist_df['bin'], ratingdiff_hist_df['cnt'], width=9,
       color='#e74c3c', edgecolor='black', linewidth=0.2, alpha=0.85)
ax.set_xlabel("ratingDiff (bin width=10)")
ax.set_ylabel("Row count")
ax.set_title(
    f'matches_raw.ratingDiff '
    f'(N={ratingdiff_non_null:,} non-NULL of {total_rows:,} total, '
    f'{ratingdiff_null_pct:.2f}% NULL; skew={ratingdiff_skewness:.4f}, '
    f'kurt={ratingdiff_kurtosis:.4f})'
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig(plots_dir / "01_02_05_ratingDiff_histogram.png", dpi=150)
plt.close()
print("Saved: 01_02_05_ratingDiff_histogram.png")

Saved: 01_02_05_ratingDiff_histogram.png


## T06 -- Plot 8: leaderboards_raw numeric boxplots (two-panel)

In [19]:
lb_stats_df = pd.DataFrame(census["leaderboards_raw_numeric_stats"])
print("=== leaderboards_raw_numeric_stats (feeds boxplots) ===")
print(lb_stats_df.to_string(index=False))
lb_zero_df = pd.DataFrame(census["leaderboards_raw_zero_counts"])
print("\n=== leaderboards_raw_zero_counts ===")
print(lb_zero_df.to_string(index=False))
lb_sk_df = pd.DataFrame(census["leaderboards_raw_skew_kurtosis"])
print("\n=== leaderboards_raw_skew_kurtosis ===")
print(lb_sk_df.to_string(index=False))

=== leaderboards_raw_numeric_stats (feeds boxplots) ===
column_name  min_val  max_val  mean_val  median_val  stddev_val    p05     p25      p75      p95
       rank        1   247604  86228.83     73999.0    61221.47 2842.0 33580.0 138103.0 190546.0
     rating     -441     3110   1038.78      1003.0      199.19  746.0   953.0   1111.0   1408.0
       wins        0    13878    107.40        27.0      275.47    4.0    10.0     84.0    467.0
     losses        0    15274    102.91        26.0      260.42    4.0    11.0     79.0    453.0
      games       10    22446    174.36        45.0      466.25   11.0    20.0    128.0    741.0
     streak    -1331     1000     -0.13         1.0        5.45   -6.0    -2.0      2.0      5.0
      drops        0     4553      5.88         1.0       22.21    0.0     0.0      4.0     24.0
rankCountry        1   105669  15300.75      3383.0    23179.72   59.0   855.0  18088.0  70491.0
     season       -1       -1     -1.00        -1.0        0.00   -1.0 

In [20]:
# I7: season excluded (constant=-1, stddev=0, no information content).
# rankLevel excluded (constant=1, skewness=-273,614,308.64 is a numerical
# artifact of zero variance, not a meaningful distribution shape).
exclude_cols = {"season", "rankLevel"}
panel_a_cols = ["rank", "rating"]
panel_b_cols = ["wins", "losses", "games", "streak", "drops", "rankCountry"]


def build_bxp_stats(row):
    """Build bxp stats dict from a row of leaderboards_raw_numeric_stats."""
    return {
        "med": float(row["median_val"]),  # median_val is the field name, NOT p50
        "q1": float(row["p25"]),
        "q3": float(row["p75"]),
        "whislo": float(row["p05"]),
        "whishi": float(row["p95"]),
        "fliers": [],
        "label": row["column_name"],
    }


def get_zero_count(col):
    """Look up zero count for a column."""
    rows = lb_zero_df[lb_zero_df['column_name'] == col]
    if rows.empty:
        return 0
    return int(rows.iloc[0]['zero_count'])


def get_skewness(col):
    """Look up skewness for a column."""
    rows = lb_sk_df[lb_sk_df['column_name'] == col]
    if rows.empty:
        return float("nan")
    return float(rows.iloc[0]['skewness'])


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
# Panel A: rating-scale columns
panel_a_data = lb_stats_df[lb_stats_df['column_name'].isin(panel_a_cols)]
bxp_a = [build_bxp_stats(r) for _, r in panel_a_data.iterrows()]
ax1.bxp(bxp_a, showfliers=False, patch_artist=True,
        boxprops=dict(facecolor='#3498db', alpha=0.7))
ax1.set_title('leaderboards_raw: Rating-Scale Columns\n(p05/p25/median/p75/p95)')
ax1.set_ylabel("Value")
for i, col in enumerate(panel_a_cols):
    z = get_zero_count(col)
    s = get_skewness(col)
    ax1.annotate(
        f'zeros={z:,}\nskew={s:.2f}',
        xy=(i + 1, ax1.get_ylim()[0]),
        ha='center', fontsize=7, color='#333333'
    )
# Panel B: activity-count columns with symlog y-axis
panel_b_data = lb_stats_df[lb_stats_df['column_name'].isin(panel_b_cols)]
bxp_b = [build_bxp_stats(r) for _, r in panel_b_data.iterrows()]
ax2.bxp(bxp_b, showfliers=False, patch_artist=True,
        boxprops=dict(facecolor='#e67e22', alpha=0.7))
ax2.set_yscale("symlog")
ax2.set_title('leaderboards_raw: Activity-Count Columns\n(symlog scale, p05/p25/median/p75/p95)')
ax2.set_ylabel("Value (symlog)")
for i, col in enumerate(panel_b_cols):
    z = get_zero_count(col)
    s = get_skewness(col)
    ax2.annotate(
        f'zeros={z:,}\nskew={s:.2f}',
        xy=(i + 1, ax2.get_ylim()[0]),
        ha='center', fontsize=7, color='#333333'
    )
plt.tight_layout()
plt.savefig(plots_dir / "01_02_05_leaderboards_boxplots.png", dpi=150)
plt.close()
print("Saved: 01_02_05_leaderboards_boxplots.png")

Saved: 01_02_05_leaderboards_boxplots.png


## T07 -- Plot 9: Completeness matrix (NULL rate bar chart for 55 matches_raw columns)

In [21]:
# Horizontal bar chart used instead of heatmap: with 55 columns and a single
# NULL-rate dimension (no row-level structure), a heatmap adds no information
# over a bar chart. The bar chart provides higher information density per
# EDA Manual Section 7 (anti cherry-picking).
null_df_plot = pd.DataFrame(census["matches_raw_null_census"])
print(f"=== matches_raw NULL rates ({len(null_df_plot)} columns) ===")
print(null_df_plot.sort_values('null_pct', ascending=False).to_string(index=False))

=== matches_raw NULL rates (55 columns) ===
          column_name  total_rows  null_count  null_pct  approx_cardinality
           modDataset   277099059   276323182     99.72                 955
             scenario   277099059   272305245     98.27               30484
               server   277099059   271529368     97.99                  11
             password   277099059   229715120     82.90                   2
        antiquityMode   277099059   190256214     68.66                   2
             hideCivs   277099059   136609836     49.30                   2
           ratingDiff   277099059   117656260     42.46                 353
               rating   277099059   117656260     42.46                4196
              country   277099059    34914481     12.60                 222
         regicideMode   277099059    20477620      7.39                   2
                 team   277099059    13577854      4.90                  31
                  won   277099059    1298556

In [22]:
null_df_sorted = null_df_plot.sort_values('null_pct', ascending=True)
# I7: <1% green (MCAR-safe per EDA Manual Section 4.5); 1-10% orange (moderate;
# warrants investigation); >10% red (severe; feature flagged for cleaning phase)
colors = []
for pct in null_df_sorted['null_pct']:
    if pct < 1:
        colors.append('#2ecc71')   # green: < 1%
    elif pct < 10:
        colors.append('#f39c12')   # orange: 1-10%
    else:
        colors.append('#e74c3c')   # red: > 10%
fig, ax = plt.subplots(figsize=(12, 16))
ax.barh(null_df_sorted['column_name'], null_df_sorted['null_pct'], color=colors)
ax.set_xlabel("NULL %")
ax.set_title("matches_raw: Column NULL Rates (55 columns)")
ax.axvline(x=1, color='gray', linestyle='--', alpha=0.5, label='1%')
ax.axvline(x=10, color='gray', linestyle=':', alpha=0.5, label='10%')
ax.legend()
plt.tight_layout()
plt.savefig(plots_dir / "01_02_05_completeness_matrix.png", dpi=150)
plt.close()
print("Saved: 01_02_05_completeness_matrix.png")

Saved: 01_02_05_completeness_matrix.png


## T08 -- Plot 10: profiles_raw NULL rates

In [23]:
profiles_null_df = pd.DataFrame(census["profiles_raw_null_census"])
print(f"=== profiles_raw NULL rates ({len(profiles_null_df)} columns) ===")
print(profiles_null_df.sort_values('null_pct', ascending=False).to_string(index=False))

=== profiles_raw NULL rates (14 columns) ===
       column_name  total_rows  null_count  null_pct  approx_cardinality
     sharedHistory     3609686     3609686    100.00                   2
     twitchChannel     3609686     3609686    100.00                  43
    youtubeChannel     3609686     3609686    100.00                   9
youtubeChannelName     3609686     3609686    100.00                   9
         discordId     3609686     3609686    100.00                  44
       discordName     3609686     3609686    100.00                  44
 discordInvitation     3609686     3609686    100.00                   6
           country     3609686     1604144     44.44                 237
        avatarhash     3609686       32487      0.90             1658324
           steamId     3609686        1805      0.05             3668366
         profileId     3609686           0      0.00             4093912
              name     3609686           0      0.00             3023244
      

In [24]:
profiles_null_sorted = profiles_null_df.sort_values('null_pct', ascending=True)
profile_colors = []
for pct in profiles_null_sorted['null_pct']:
    if pct >= 100:
        profile_colors.append('#e74c3c')   # red: dead column
    else:
        profile_colors.append('#3498db')   # blue: live column
fig, ax = plt.subplots(figsize=(10, max(5, len(profiles_null_sorted) * 0.4)))
bars = ax.barh(
    profiles_null_sorted['column_name'],
    profiles_null_sorted['null_pct'],
    color=profile_colors,
    edgecolor='black', linewidth=0.3
)
# Annotate dead bars
for bar, pct, col in zip(bars, profiles_null_sorted['null_pct'],
                         profiles_null_sorted['column_name']):
    if pct >= 100:
        ax.text(
            bar.get_width() / 2, bar.get_y() + bar.get_height() / 2,
            'DEAD', ha='center', va='center', fontsize=8,
            fontweight='bold', color='white'
        )
ax.set_xlabel("NULL %")
ax.set_title("profiles_raw: Column NULL Rates (red=DEAD, blue=live)")
ax.axvline(x=100, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(plots_dir / "01_02_05_profiles_null_rates.png", dpi=150)
plt.close()
print("Saved: 01_02_05_profiles_null_rates.png")
dead_cols = profiles_null_sorted[profiles_null_sorted['null_pct'] >= 100]['column_name'].tolist()
print(f"Dead columns ({len(dead_cols)}): {dead_cols}")

Saved: 01_02_05_profiles_null_rates.png
Dead columns (7): ['sharedHistory', 'twitchChannel', 'youtubeChannel', 'youtubeChannelName', 'discordId', 'discordName', 'discordInvitation']


## T09 -- Plot 11: leaderboards_raw.leaderboard top-k

In [25]:
lb_leaderboard = pd.DataFrame(census["leaderboards_raw_categorical"]["leaderboard"])
print(f"=== leaderboards_raw.leaderboard ({len(lb_leaderboard)} values) ===")
print(lb_leaderboard.to_string(index=False))

=== leaderboards_raw.leaderboard (16 values) ===
                       value     cnt   pct
                    unranked 1729627 72.64
                     rm_team  329438 13.83
                      rm_1v1  242054 10.17
                     ew_team   29593  1.24
                      ew_1v1   17061  0.72
             rm_team_console   12938  0.54
              rm_1v1_console    8935  0.38
                     unknown    7118  0.30
        ew_1v1_redbullwololo    3599  0.15
                     ror_1v1     357  0.01
                    ror_team     340  0.01
                     dm_team      81  0.00
                      dm_1v1      38  0.00
              ew_1v1_console      25  0.00
             ew_team_console      21  0.00
ew_1v1_redbullwololo_console       2  0.00


In [26]:
lb_leaderboard_sorted = lb_leaderboard.sort_values('cnt', ascending=True)
fig, ax = plt.subplots(figsize=(10, max(4, len(lb_leaderboard_sorted) * 0.45)))
bars = ax.barh(lb_leaderboard_sorted['value'], lb_leaderboard_sorted['cnt'],
               color='#2980b9', edgecolor='black', linewidth=0.3)
for bar, cnt in zip(bars, lb_leaderboard_sorted['cnt']):
    ax.text(
        bar.get_width() + bar.get_width() * 0.01, bar.get_y() + bar.get_height() / 2,
        f'{cnt:,}', va='center', ha='left', fontsize=9
    )
ax.set_xlabel("Row count")
ax.set_title("leaderboards_raw: leaderboard Distribution")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig(plots_dir / "01_02_05_lb_leaderboard_distribution.png", dpi=150)
plt.close()
print("Saved: 01_02_05_lb_leaderboard_distribution.png")

Saved: 01_02_05_lb_leaderboard_distribution.png


## T10 -- Plot 12: Boolean columns stacked bar

In [27]:
bool_df = pd.DataFrame(census["boolean_census"])
print(f"=== boolean_census ({len(bool_df)} columns) ===")
print(bool_df.to_string(index=False))

=== boolean_census (18 columns) ===
      column_name  true_count  false_count  null_count
              mod     7022020    270077039           0
     fullTechTree     1074868    275592697      431494
      allowCheats     2830361    273840360      428338
   empireWarsMode   255573532     18595385     2930142
        lockSpeed   251301799     25369495      427765
        lockTeams    32135660    244535509      427890
         hideCivs     4991659    135498344   136609056
       recordGame   255765951     20905084      428024
     regicideMode   240124881     16485894    20488284
sharedExploration    73294670    203377119      427270
  suddenDeathMode   256318276     17891014     2889769
    antiquityMode    72382175     14468333   190248551
    teamPositions   228639034     48031894      428131
     teamTogether   272146075      4524654      428330
        turboMode   251931724     24739523      427812
         password    39902054      7491234   229705771
           shared      531363

In [28]:
total = census["matches_raw_total_rows"]
bool_df['true_pct'] = 100.0 * bool_df['true_count'] / total
bool_df['false_pct'] = 100.0 * bool_df['false_count'] / total
bool_df['null_pct_calc'] = 100.0 * bool_df['null_count'] / total
fig, ax = plt.subplots(figsize=(12, 8))
y = range(len(bool_df))
ax.barh(y, bool_df['true_pct'], color='#2ecc71', label='TRUE')
ax.barh(y, bool_df['false_pct'], left=bool_df['true_pct'],
        color='#e74c3c', label='FALSE')
ax.barh(y, bool_df['null_pct_calc'],
        left=bool_df['true_pct'] + bool_df['false_pct'],
        color='#95a5a6', label='NULL')
ax.set_yticks(list(y))
ax.set_yticklabels(bool_df['column_name'])
ax.set_xlabel("Percentage")
ax.set_title("Boolean Column Proportions -- matches_raw")
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(plots_dir / "01_02_05_boolean_stacked.png", dpi=150)
plt.close()
print(f"Saved: 01_02_05_boolean_stacked.png ({len(bool_df)} boolean columns)")

Saved: 01_02_05_boolean_stacked.png (18 boolean columns)


## T11 -- Plot 13: Monthly match volume line chart

In [29]:
monthly_df = con.execute('''
    SELECT
        DATE_TRUNC('month', started) AS month,
        COUNT(DISTINCT matchId) AS distinct_matches,
        COUNT(*) AS total_rows
    FROM matches_raw
    WHERE started IS NOT NULL
    GROUP BY month
    ORDER BY month
''').df()
print("=== monthly match volume (feeds line chart) ===")
print("Head:")
print(monthly_df.head(5).to_string(index=False))
print("Tail:")
print(monthly_df.tail(5).to_string(index=False))

=== monthly match volume (feeds line chart) ===
Head:
     month  distinct_matches  total_rows
2020-07-01               135         665
2020-08-01            149946      679650
2020-09-01            147402      664976
2020-10-01            146176      651647
2020-11-01            241340     1087742
Tail:
     month  distinct_matches  total_rows
2025-12-01           1493950     5305076
2026-01-01           1648162     5830648
2026-02-01           1422625     5153218
2026-03-01           1495452     5383385
2026-04-01            197194      712834


In [30]:
ratings_start = census["temporal_range_ratings"][0]["earliest_rating"]
print(f"ratings_raw earliest_rating (reference line): {ratings_start}")
monthly_df['month'] = pd.to_datetime(monthly_df['month'])
ratings_start_dt = pd.to_datetime(ratings_start)
fig, ax1 = plt.subplots(figsize=(14, 6))
color_matches = '#2980b9'
color_rows = '#e74c3c'
ax1.plot(monthly_df['month'], monthly_df['distinct_matches'],
         color=color_matches, linewidth=1.5, label='distinct_matches')
ax1.set_xlabel("Month")
ax1.set_ylabel("Distinct Matches", color=color_matches)
ax1.tick_params(axis='y', labelcolor=color_matches)
ax2 = ax1.twinx()
ax2.plot(monthly_df['month'], monthly_df['total_rows'],
         color=color_rows, linewidth=1.5, linestyle='--', label='total_rows')
ax2.set_ylabel("Total Rows", color=color_rows)
ax2.tick_params(axis='y', labelcolor=color_rows)
ax1.axvline(x=ratings_start_dt, color='purple', linestyle=':', linewidth=1.5,
            label='ratings_raw start')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.set_title("Monthly Match Volume -- matches_raw")
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(plots_dir / "01_02_05_monthly_volume.png", dpi=150)
plt.close()
print("Saved: 01_02_05_monthly_volume.png")
print(f"Time range: {monthly_df['month'].min()} to {monthly_df['month'].max()}")

ratings_raw earliest_rating (reference line): 2022-09-10 00:00:00
Saved: 01_02_05_monthly_volume.png
Time range: 2020-07-01 00:00:00 to 2026-04-01 00:00:00


## T12 -- Write markdown summary artifact and close

In [31]:
sql_queries = {
    "rating_histogram": (
        "SELECT (FLOOR(rating / 100) * 100)::INTEGER AS bin, COUNT(*) AS cnt\n"
        "FROM matches_raw WHERE rating IS NOT NULL\n"
        "GROUP BY bin ORDER BY bin"
    ),
    "ratingDiff_histogram": (
        'SELECT (FLOOR(\"ratingDiff\" / 10) * 10)::INTEGER AS bin, COUNT(*) AS cnt\n'
        'FROM matches_raw WHERE \"ratingDiff\" IS NOT NULL\n'
        "GROUP BY bin ORDER BY bin"
    ),
    "monthly_volume": (
        "SELECT\n"
        "    DATE_TRUNC('month', started) AS month,\n"
        "    COUNT(DISTINCT matchId) AS distinct_matches,\n"
        "    COUNT(*) AS total_rows\n"
        "FROM matches_raw\n"
        "WHERE started IS NOT NULL\n"
        "GROUP BY month\n"
        "ORDER BY month"
    ),
}
plots_info = [
    ("01_02_05_won_distribution.png", "Plot 1: won Distribution Bar Chart",
     "Vertical bar chart showing TRUE/FALSE/NULL counts for matches_raw.won; "
     "reveals ~4.69% NULL rate on the prediction target."),
    ("01_02_05_won_consistency.png", "Plot 2: Intra-Match won Consistency Stacked Bar",
     f"Stacked bar showing intra-match consistency categories for 2-player matches; "
     f"~{inconsistent_pct:.2f}% of matches have both_true or both_false labels."),
    ("01_02_05_leaderboard_distribution.png", "Plot 3: matches_raw.leaderboard Distribution",
     "Horizontal bar chart of leaderboard values in matches_raw."),
    ("01_02_05_civ_top30.png", "Plot 4: matches_raw.civ Top 30",
     "Horizontal bar chart of top-30 civilization values in matches_raw."),
    ("01_02_05_map_top30.png", "Plot 5: matches_raw.map Top 30",
     "Horizontal bar chart of top-30 map values in matches_raw."),
    ("01_02_05_rating_histogram.png", "Plot 6: matches_raw.rating Histogram",
     "Histogram with bin_width=100 showing rating distribution; skew=0.5662, kurt=1.6157."),
    ("01_02_05_ratingDiff_histogram.png", "Plot 7: matches_raw.ratingDiff Histogram",
     "Histogram with bin_width=10 showing ratingDiff distribution; skew=0.1105, kurt=0.8900."),
    ("01_02_05_leaderboards_boxplots.png", "Plot 8: leaderboards_raw Numeric Boxplots (two-panel)",
     "Two-panel boxplots (rating-scale and activity-count with symlog scale); "
     "season and rankLevel excluded as constants."),
    ("01_02_05_completeness_matrix.png", "Plot 9: matches_raw Completeness Matrix (55 columns)",
     "Horizontal bar chart of NULL % for all 55 matches_raw columns, "
     "color-coded green/orange/red by severity."),
    ("01_02_05_profiles_null_rates.png", "Plot 10: profiles_raw NULL Rates",
     "Horizontal bar chart distinguishing 100% NULL (dead) columns from live columns in profiles_raw."),
    ("01_02_05_lb_leaderboard_distribution.png",
     "Plot 11: leaderboards_raw.leaderboard Distribution",
     "Horizontal bar chart of leaderboard type counts in leaderboards_raw."),
    ("01_02_05_boolean_stacked.png", "Plot 12: matches_raw Boolean Column Proportions",
     "Stacked bar chart showing TRUE/FALSE/NULL proportions for all 18 boolean columns."),
    ("01_02_05_monthly_volume.png", "Plot 13: Monthly Match Volume",
     "Dual-axis line chart of monthly distinct match count and total rows; "
     "vertical reference line marks ratings_raw start (2022-09-10)."),
]
md_lines = [
    "# Step 01_02_05 -- Univariate Census Visualizations: aoe2companion",
    "",
    "**Generated by:** `sandbox/aoe2/aoe2companion/01_exploration/02_eda/01_02_05_visualizations.py`",
    "**Source artifact:** `01_02_04_univariate_census.json`",
    "**Plots directory:** `artifacts/01_exploration/02_eda/plots/`",
    "",
    "## Plots",
    "",
]
for filename, title, caption in plots_info:
    md_lines.append(f"### {title}")
    md_lines.append("")
    md_lines.append(f"**File:** `plots/{filename}`")
    md_lines.append("")
    md_lines.append(f"**Caption:** {caption}")
    md_lines.append("")
md_lines += [
    "## SQL Queries (Invariant #6 -- Reproducibility)",
    "",
    "All DuckDB SQL queries used to produce plots in this notebook are listed verbatim below.",
    "Any plot produced from a DuckDB query can be reproduced by running the query against",
    "the persistent `aoe2companion` DuckDB.",
    "",
    "### rating histogram (Plot 6)",
    "",
    "```sql",
    sql_queries["rating_histogram"],
    "```",
    "",
    "### ratingDiff histogram (Plot 7)",
    "",
    "```sql",
    sql_queries["ratingDiff_histogram"],
    "```",
    "",
    "### monthly match volume (Plot 13)",
    "",
    "```sql",
    sql_queries["monthly_volume"],
    "```",
    "",
]
md_content = '\n'.join(md_lines)
md_path = artifacts_dir / "01_02_05_visualizations.md"
with open(md_path, 'w') as f:
    f.write(md_content)
print(f"Written: {md_path}")
print("\n=== Plot file verification ===")
all_ok = True
for filename, title, _ in plots_info:
    png_path = plots_dir / filename
    exists = png_path.exists()
    status = "OK" if exists else "MISSING"
    print(f"  {status}: {filename}")
    if not exists:
        all_ok = False
print(f"\nAll 13 plots present: {all_ok}")

Written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/02_eda/01_02_05_visualizations.md

=== Plot file verification ===
  OK: 01_02_05_won_distribution.png
  OK: 01_02_05_won_consistency.png
  OK: 01_02_05_leaderboard_distribution.png
  OK: 01_02_05_civ_top30.png
  OK: 01_02_05_map_top30.png
  OK: 01_02_05_rating_histogram.png
  OK: 01_02_05_ratingDiff_histogram.png
  OK: 01_02_05_leaderboards_boxplots.png
  OK: 01_02_05_completeness_matrix.png
  OK: 01_02_05_profiles_null_rates.png
  OK: 01_02_05_lb_leaderboard_distribution.png
  OK: 01_02_05_boolean_stacked.png
  OK: 01_02_05_monthly_volume.png

All 13 plots present: True


In [32]:
con.close()
print("DB connection closed.")

DB connection closed.
